# Maya 2 voice server on a free Colab GPU
Makes Studio and live conversation speak in the cloned voice with Maya 2 (English, Hindi, Hinglish), first sound in about 0.13 s.

1. **Runtime → Change runtime type → T4 GPU**, Save.
2. Run the cells in order. The last cell prints two lines: paste them into `apps/web/.env.local` **and** `apps/voice/.env.local` on your Mac, then restart the web app and the voice service.

Keep this tab open while you use it; Colab stops it after a while of inactivity, and the link changes every time. Recordings of staff voices are sent here when a voice is first used: use your own Google account and only voices whose owners agreed. For testing, not for client calls.

In [ ]:
!nvidia-smi | head -15

In [ ]:
!pip -q install "maya3[server] @ git+https://github.com/MayaResearch/maya2-native" aiohttp
# Installing maya3 moves torch to 2.8.0, which breaks the torchvision Colab ships with (the server then dies with "operator torchvision::nms does not exist"). Put a matching pair back.
!pip -q install "torchvision==0.23.0" "torchaudio==2.8.0" --index-url https://download.pytorch.org/whl/cu126
import torch, torchvision; print(torch.__version__, torchvision.__version__)

In [ ]:
import subprocess, time, urllib.request
subprocess.run(['pkill', '-f', 'maya3.server']); time.sleep(3)  # an older copy would hold port 8090
server = subprocess.Popen(['python', '-m', 'maya3.server', '--ckpt', 'maya-research/maya2-native', '--host', '127.0.0.1', '--port', '8090', '--max-seats', '2'], stdout=open('/content/maya.log', 'w'), stderr=subprocess.STDOUT)
for i in range(180):
    try:
        if urllib.request.urlopen('http://127.0.0.1:8090/health', timeout=2).status == 200: print('Maya 2 ready'); break
    except Exception: time.sleep(5)
else:
    print(open('/content/maya.log').read()[-3000:])
print(open('/content/maya.log').read()[-1500:])

The Maya 2 server has no login of its own, so this small gate only lets requests with the secret token through.

In [ ]:
import secrets, asyncio, threading
from aiohttp import web, ClientSession
TOKEN = secrets.token_urlsafe(32)
async def gate(request):
    if request.headers.get('authorization') != f'Bearer {TOKEN}':
        return web.Response(status=401, text='bad token')
    body = await request.read()
    async with ClientSession() as s:
        async with s.request(request.method, f'http://127.0.0.1:8090{request.path_qs}', data=body or None,
                             headers={k: v for k, v in request.headers.items() if k.lower() in ('content-type',)}) as r:
            resp = web.StreamResponse(status=r.status, headers={k: v for k, v in r.headers.items() if k.lower() in ('content-type', 'x-sample-rate', 'x-encoding')})
            await resp.prepare(request)
            async for chunk in r.content.iter_any(): await resp.write(chunk)
            await resp.write_eof(); return resp
async def start():
    app = web.Application(client_max_size=64 * 1024 * 1024); app.router.add_route('*', '/{tail:.*}', gate)
    runner = web.AppRunner(app); await runner.setup(); await web.TCPSite(runner, '127.0.0.1', 8081).start()
def run():
    loop = asyncio.new_event_loop(); loop.run_until_complete(start()); loop.run_forever()
threading.Thread(target=run, daemon=True).start(); time.sleep(2); print('gate on 8081')
import urllib.request as _u
_r = _u.Request('http://127.0.0.1:8081/v1/voices', headers={'authorization': f'Bearer {TOKEN}'})
print('self-check through the gate:', _u.urlopen(_r, timeout=10).status, '(200 means Maya 2 is answering)')

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
import re
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8081'], stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT)
url = None
for i in range(60):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m: url = m.group(0); break
    time.sleep(2)
print()
print('Paste these two lines into apps/web/.env.local AND apps/voice/.env.local:')
print()
print(f'MAYA2_URL={url}')
print(f'MAYA2_TOKEN={TOKEN}')

## Hindi and Hinglish speech-to-text: IndicConformer on the same GPU
Run these two cells after the Maya 2 ones. The model is gated on Hugging Face: sign in with the account that accepted the terms at https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual (a box appears for the token). The last line printed is `INDIC_STT_URL=...`: add it to `apps/voice/.env.local` and restart the voice service. English calls keep using local Whisper.
It runs as its own process with the GPU build of onnxruntime (version 1.22: newer ones refuse the way Hugging Face stores the model files). After the first request (about 20 s to warm up) the model takes 0.1 to 0.4 s per phrase.
Sign out of Hugging Face in this session when you finish (`from huggingface_hub import logout; logout()`).

In [ ]:
from huggingface_hub import login
login()
!pip -q install soundfile librosa
!rm -rf /content/ortgpu && pip -q install --target /content/ortgpu --no-deps "onnxruntime-gpu==1.22.0" 2>&1 | tail -2
import secrets
open('/content/stt_secret.txt', 'w').write(secrets.token_urlsafe(24))
open('/content/stt_server.py', 'w').write(r'''
import sys; sys.path.insert(0, "/content/ortgpu")
import onnxruntime as ort
try: ort.preload_dlls()
except Exception as e: print("preload:", e)
print("providers:", ort.get_available_providers(), flush=True)
import io, time, asyncio, torch, soundfile as sf, librosa
from aiohttp import web
from transformers import AutoModel
model = AutoModel.from_pretrained("ai4bharat/indic-conformer-600m-multilingual", trust_remote_code=True)
def stt(b):
    a, sr = sf.read(io.BytesIO(b), dtype="float32")
    if a.ndim > 1: a = a.mean(1)
    if sr != 16000: a = librosa.resample(a, orig_sr=sr, target_sr=16000)
    with torch.no_grad(): return str(model(torch.from_numpy(a).unsqueeze(0), "hi", "ctc")).strip()
SECRET = open("/content/stt_secret.txt").read().strip()
lock = asyncio.Lock()
async def inference(request):
    if request.match_info["secret"] != SECRET: return web.Response(status=401, text="bad token")
    form = await request.post(); data = form["file"].file.read()
    async with lock:
        t = time.time(); text = await asyncio.get_running_loop().run_in_executor(None, stt, data)
    return web.json_response({"text": text, "ms": int((time.time()-t)*1000)})
app = web.Application(client_max_size=64*1024*1024); app.router.add_post("/{secret}/inference", inference)
print("serving", flush=True)
web.run_app(app, host="127.0.0.1", port=8082, print=None)
''')

In [ ]:
import glob, shutil
for q in glob.glob('/content/ortgpu/numpy*') + glob.glob('/content/ortgpu/protobuf*') + glob.glob('/content/ortgpu/google'):
    shutil.rmtree(q, ignore_errors=True)   # keep only the GPU onnxruntime from that folder
subprocess.run(['pkill', '-f', 'stt_server.py']); time.sleep(1)
stt_proc = subprocess.Popen(['python', '/content/stt_server.py'], stdout=open('/content/stt.log', 'w'), stderr=subprocess.STDOUT)
for i in range(120):
    time.sleep(3)
    log = open('/content/stt.log').read()
    if 'serving' in log or stt_proc.poll() is not None: break
print(log[-800:])
STT_SECRET = open('/content/stt_secret.txt').read().strip()
t2 = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8082'], stdout=open('/content/tunnel2.log', 'w'), stderr=subprocess.STDOUT)
u2 = None
for i in range(60):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel2.log').read())
    if m: u2 = m.group(0); break
    time.sleep(2)
print()
print(f'INDIC_STT_URL={u2}/{STT_SECRET}')